# Modelos del lenguaje de `ngramas`

In [1]:
from collections import Counter, defaultdict
from itertools import chain
import re
import pandas as pd
import numpy as np
from rich import print as rprint
from sklearn.model_selection import train_test_split

In [2]:
def vocab() -> defaultdict:
    """Crea diccionario de vocabulario

    Asigna a cada palabra un índice numerico
    único

    Return
    ------
    defaultdict
        Diccionario con el vocabulario
    """
    vocab = defaultdict()
    vocab.default_factory = lambda: len(vocab)
    return vocab

In [3]:
my_vocab = vocab()

In [4]:
my_vocab["hola"]

0

In [5]:
rprint(dict(my_vocab))

{'hola': 0}

In [6]:
def text2number(corpus: list[str], vocab: defaultdict) -> list[int]:
    """Convierte una cadena de simbolos a una secuencia numerica

    Parameters
    ----------
    corpus: list
        Lista con oraciones
    vocab: defaultdict
        Diccionario que asigna indices únicos por cada palabra

    Return
    ------
    list[int]:
        Lista con indices de palabras
    """
    result = []
    for sent in corpus:
        result.append([vocab[word] for word in sent.split()])
    return result

In [7]:
toy_corpus = [
    "que bueno que te encuentro quiero decirte hola",
    "la buena aventura",
    "el melocoton dice hola"
]

In [8]:
idx_corpus = text2number(toy_corpus, my_vocab)
for row in idx_corpus:
    rprint(row)

[1, 2, 1, 3, 4, 5, 6, 0]

[7, 8, 9]

[10, 11, 12, 0]

In [9]:
def get_invert_vocab(vocab: defaultdict) -> dict:
    return {idx: word for word, idx in vocab.items()}

In [10]:
words_idx = get_invert_vocab(my_vocab)
rprint(words_idx)

{
    0: 'hola',
    1: 'que',
    2: 'bueno',
    3: 'te',
    4: 'encuentro',
    5: 'quiero',
    6: 'decirte',
    7: 'la',
    8: 'buena',
    9: 'aventura',
    10: 'el',
    11: 'melocoton',
    12: 'dice'
}

In [11]:
def number2text(corpus: list[int], vocab: dict) -> list[str]:
    result = []
    for word_indices in corpus:
        result.append([vocab[idx] for idx in word_indices])
    return result

In [12]:
number2text(idx_corpus, words_idx)

[['que', 'bueno', 'que', 'te', 'encuentro', 'quiero', 'decirte', 'hola'],
 ['la', 'buena', 'aventura'],
 ['el', 'melocoton', 'dice', 'hola']]

## Corpus

In [29]:
!pip install -U elotl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 5.5 MB/s eta 0:00:0017.4 MB/s eta 0:00:01
  Attempting uninstall: elotl
    Found existing installation: elotl 0.0.1.16
    Uninstalling elotl-0.0.1.16:
      Successfully uninstalled elotl-0.0.1.16


In [15]:
import elotl.corpus

kolo = pd.DataFrame(elotl.corpus.load("kolo"), columns=["l1", "l2", "variant", "doc"])

In [16]:
kolo.head()

,l1,l2,variant,doc
0,Si el clarín con su bélico acento,"Ta na ndei'i trompeta xi'in tachi ra,",Mixteco de Tezoatlán (mxb),El Himno Nacional
1,Cómo alimentar a los niños,Jā nāsa kajī sūchī lúlí,Mixteco de Santo Tomás Ocotepec (mie),Como curar algunas enfermedades
2,el Señor ocho Venado,Iꞌà Nacuaa,Mixteco de Chalcatongo (mig),Curso de lengua Mixteca
3,Las gallinas ponen huevos,Nsìvì nduá sakín sìùn.,Mixteco San Jerónimo Xayacatlán (mit),Abecedario Mixteco
4,Dicen que la gente que se cae en una barranca ...,"Ñayii ka ke koo kava ma te ka ji'i ma, chi yuk...",Mixteco de Magdalena Peñasco (xtm),Algunos dichos y creencias tradicionales de Ma...


In [18]:
rprint(kolo.groupby("doc")["doc"].count())

doc
Abecedario Mixteco                                                53
Algunos dichos y creencias tradicionales de Magdalena Peñasco    272
Bichos                                                            61
Burro lindo                                                       17
Como curar algunas enfermedades                                  327
Cuentos y ejercicios en español y mixteco                        124
Curso de lengua Mixteca                                          504
Cómo leer en Mixteco                                             150
Del maíz a la tortilla                                             8
El Himno Nacional                                                 27
El cuento de una mujer y su marrano                               90
La canción del gato                                               45
La cucaracha, la gallina, el coyote y el señor                    32
Name: doc, dtype: int64

In [19]:
corpus = list(kolo[kolo["doc"] == "El cuento de una mujer y su marrano"]["l1"])

In [20]:
len(corpus)

90

In [21]:
rprint(len(corpus))
rprint(corpus[:3])

90

[
    'El perro comenzó a morder al marrano',
    '—¿Qué compraré con esta moneda chica? Se dijo a sí misma.',
    'El agua no quiere apagar al fuego.'
]

## Preprocesamiento

In [22]:
def preprocess(words: list, regex="\w+") -> list:
    lower_words = list(map(str.lower, words))
    return [
        re.sub(r"[^\w\s]", "", word)
        for word in lower_words
    ]

In [23]:
rprint(preprocess(corpus[:4]))

[
    'el perro comenzó a morder al marrano',
    'qué compraré con esta moneda chica se dijo a sí misma',
    'el agua no quiere apagar al fuego',
    'en el camino encontró un corral'
]

In [24]:
corpus = preprocess(corpus)

In [25]:
train_data, test_data = train_test_split(corpus, test_size=0.3)

In [26]:
rprint(len(train_data), len(test_data))

63 27

## Construyendo un vocabulario indexado

In [19]:
kolo_vocab = vocab()

In [20]:
kolo_indices = list(text2number(train_data, kolo_vocab))

In [21]:
rprint(kolo_indices[:3])

[[0, 1, 2, 3, 4, 5, 6], [7, 8, 2, 3, 9, 10, 11, 12, 2, 3, 13, 14, 0, 1], [0, 1, 2, 3, 4, 5, 6]]

In [22]:
rprint(dict(kolo_vocab))

{
    'mi': 0,
    'marrano': 1,
    'no': 2,
    'quiere': 3,
    'brincar': 4,
    'el': 5,
    'corral': 6,
    'la': 7,
    'vara': 8,
    'pegar': 9,
    'al': 10,
    'perroel': 11,
    'perro': 12,
    'morder': 13,
    'a': 14,
    'por': 15,
    'que': 16,
    'ratón': 17,
    'roer': 18,
    'lazo': 19,
    'fuego': 20,
    'quemar': 21,
    'toro': 22,
    'bebo': 23,
    'agua': 24,
    'porque': 25,
    'apagar': 26,
    'utale': 27,
    'este': 28,
    'gritó': 29,
    'y': 30,
    'brincó': 31,
    'tomar': 32,
    'entonces': 33,
    'mujer': 34,
    'vió': 35,
    'un': 36,
    'le': 37,
    'dijo': 38,
    'apaga': 39,
    'hombre': 40,
    'comenzó': 41,
    'amarrar': 42,
    'dejó': 43,
    'encontró': 44,
    'yo': 45,
    'cogeré': 46,
    'gato': 47,
    'coje': 48,
    'llegaré': 49,
    'casa': 50,
    'esta': 51,
    'tarde': 52,
    'pega': 53,
    'pero': 54,
    'quiso': 55,
    'ahorcar': 56,
    'señor': 57,
    'amarre': 58,
    'ahorca': 59,
    'había': 60,
    'una': 61,
    'barrer': 62,
    'su': 63,
    'moneda': 64,
    'chica': 65,
    'está': 66,
    'bien': 67,
    'vaca': 68,
    'dame': 69,
    'manojo': 70,
    'de': 71,
    'zacate': 72,
    'te': 73,
    'daré': 74,
    'leche': 75,
    'creo': 76,
    'compraré': 77,
    'marranito': 78,
    'ella': 79,
    'fué': 80,
    'donde': 81,
    'estaba': 82,
    'dió': 83,
    'luego': 84,
    'seguir': 85,
    'cuento': 86,
    'se': 87,
    'terminó': 88,
    'siguió': 89,
    'poco': 90,
    'más': 91,
    'en': 92,
    'camino': 93,
    'vengo': 94,
    'me': 95,
    'des': 96,
    'plato': 97,
    'hubo': 98,
    'terminado': 99,
    'trabajo': 100,
    'mercado': 101,
    'compró': 102,
    'blanco': 103,
    'muy': 104,
    'bueno': 105,
    'comido': 106,
    'para': 107,
    'bebió': 108,
    'pronto': 109,
    'lamió': 110,
    'los': 111,
    'bigotes': 112,
    'muerde': 113,
    'así': 114,
    'como': 115,
    'llegó': 116,
    'ese': 117,
    'díaa': 118,
    'morderlo': 119
}

### ¿Nos falta agregar algo?


Para completar el vocabulario, agregaremos los símbolos de *BOS (Beginning Of String)* y *EOS (End Of S\
String)*. Estos simbolos estarán al inicio y final de cada cadena del conjunto de entrenamiento de tal forma que las cadenas tengan la forma siguiente:

$$<BOS> w_1 ... w_k <EOS>$$

De esta forma, podremos obtener probabilidades inciales y transiciones terminales (aquellas que van hacía el símbolo de termino EOS).

Por último, obtenemos el diccionario que nos servirá para recuperar las palabras dado los índices de este vocabulario

In [23]:
#Indicamos las etiquetas a usar
EOS = '<EOS>'
BOS = '<BOS>'

#Cada etiqeuta se le asigna un indice numerico
BOS_IDX = max(kolo_vocab.values()) + 2
EOS_IDX = max(kolo_vocab.values()) + 1

# Se agregan estas etiquetas al vocabulario
kolo_vocab[EOS] = EOS_IDX
kolo_vocab[BOS] = BOS_IDX

# A cada cadena se le agrega el índice de la etiqueta BOS al inicio y EOS al final
kolo_indices = [[BOS_IDX] + idx_sent + [EOS_IDX] for idx_sent in kolo_indices]

#Diccionario de índice : palabra
kolo_words_idx = get_invert_vocab(kolo_vocab)

In [24]:
rprint(kolo_indices[:3])
rprint(len(kolo_indices))
rprint("EOS IDX=", EOS_IDX, "BOS_IDX=", BOS_IDX)

[
    [121, 0, 1, 2, 3, 4, 5, 6, 120],
    [121, 7, 8, 2, 3, 9, 10, 11, 12, 2, 3, 13, 14, 0, 1, 120],
    [121, 0, 1, 2, 3, 4, 5, 6, 120]
]

63

EOS IDX= 120 BOS_IDX= 121

In [25]:
rprint(number2text(kolo_indices[:3], kolo_words_idx))

[
    ['<BOS>', 'mi', 'marrano', 'no', 'quiere', 'brincar', 'el', 'corral', '<EOS>'],
    [
        '<BOS>',
        'la',
        'vara',
        'no',
        'quiere',
        'pegar',
        'al',
        'perroel',
        'perro',
        'no',
        'quiere',
        'morder',
        'a',
        'mi',
        'marrano',
        '<EOS>'
    ],
    ['<BOS>', 'mi', 'marrano', 'no', 'quiere', 'brincar', 'el', 'corral', '<EOS>']
]

## Estimación del modelo de `n-gramas`

### Ejercicio: Construye la función `get_ngrams(sentences, n)`

**NOTA** No se vale usar `from nltk import ngrams` puro python estandar alv (a la viva)

In [33]:
def get_ngrams(sentences: list, n: int) -> chain:
    return chain(*[zip(*[sent[i:] for i in range(n)]) for sent in sentences])

In [ ]:
from itertools import chain
def get_ngrams_explicit(sentences, n):
    all_ngrams = []

    # Process each sentence separately
    for sentence in sentences:
        # Create n different slices of the sentence
        slices = []
        for i in range(n):
            slices.append(sentence[i:])

        # Zip the slices together to form n-grams
        sentence_ngrams = list(zip(*slices))
        all_ngrams.append(sentence_ngrams)

    # Flatten the list of n-grams from all sentences
    return list(chain(*all_ngrams))

# Example usage
sentences = ["hello world", "example text"]
bigrams = get_ngrams_explicit(sentences, 2)
print(bigrams)

[('h', 'e'), ('e', 'l'), ('l', 'l'), ('l', 'o'), ('o', ' '), (' ', 'w'), ('w', 'o'), ('o', 'r'), ('r', 'l'), ('l', 'd'), ('e', 'x'), ('x', 'a'), ('a', 'm'), ('m', 'p'), ('p', 'l'), ('l', 'e'), ('e', ' '), (' ', 't'), ('t', 'e'), ('e', 'x'), ('x', 't')]


In [36]:
list(get_ngrams([[1, 2, 3], [4, 5, 6]], n=2))

[(1, 2), (2, 3), (4, 5), (5, 6)]

### Modelo de lenguaje


Una vez preprocesadas las cadenas pasaremos a estimar el modelo. Para esta estimación, tomaremos en cuenta dos parámetros:

*   El tamaño de n-gramas; es decir, qué tantos elementos previos consideraremos para estimar la probabilidad de que ocurra una palabra.
    - bigramas
    - trigramas
    - etc
*   El elemento $\lambda$ para estimar la probabilidad con smoothing de Lidstone. En ese sentido, dado un n-grama $w_{i-n+1} ... w_{i-1} w_i$ estimaremos la probabilidad como:

$$p(w_i|w_{i-1}...w_{i-n+1}) = \frac{C(w_{i-n+1} ... w_{i-1} w_i) + \lambda}{C(w_{i-n+1} ... w_{i-1}) + \lambda V}$$

donde $V$ es el tamaño del vocabulario.

In [28]:
def get_model(corpus: list, vocab: defaultdict, n: int=2, l: float=1.0):
    ngrams = get_ngrams(corpus, n)

    freq_grams = Counter(ngrams)
    # Obtenermos el tamaño del vocabulario
    V = len(vocab) - 2

    # Calculo de la dimensión del tensor de transiciones
    # En palabras condicionadas consideraremos al elemento EOS
    dim = (V,) * (n - 1) + (V + 1,)
    # Tensor de transiciones
    A = np.zeros(dim)
    # Probabilidades iniciales
    Pi = np.zeros(V)

    # Calculo de frecuencias
    for ngram, freq in freq_grams.items():
        # Llenado del tensor de transiciones
        if ngram[0] != BOS_IDX:
            A[ngram] = freq
        # Llenado de frecuencias iniciales
        elif ngram[0] == BOS_IDX and ngram[1] != EOS_IDX:
            Pi[ngram[1]] = freq

    # Calculo de probabilidades a partir de frecuencias
    # El parámetro l es para smoothing
    for i, b in enumerate(A):
        A[i] = (
            (b + l).T /
            (b + l).sum(n - 2)
        ).T

    # Calculo de probabilidades iniciales
    Pi = (Pi + l) / (Pi + l).sum(0)

    return A, Pi

### Detalles de la implementación

In [37]:
bigrams = list(get_ngrams(kolo_indices, n=2))
bigrams[:3]

[(121, 0), (0, 1), (1, 2)]

In [ ]:
for i, b in enumerate(bigrams):
    print(b)
    print(kolo_words_idx[b[0]], kolo_words_idx[b[1]])
    if i == 10:
        break

In [ ]:
V = len(kolo_vocab) - 2
n = 2
dim = (V,)*(n - 1) + (V + 1,)

In [ ]:
dim

(108, 109)

### Estimación de modelo de bigramas con l = 1

In [31]:
bigram_model = get_model(kolo_indices, kolo_vocab, n=2, l=1.0)

In [34]:
A_bigram = bigram_model[0]
print("Tensor dimention", A_bigram.shape)
print("Suma de probabilidades")
print(A_bigram.sum(1))

Tensor dimention (120, 121)
Suma de probabilidades
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [70]:
A_bigram.shape

(120, 121)

## Aplicaciones

1. Obtener la probabilidad de una cadena
2. Predecir una palabra siguiente
3. Generación de texto

Para determinar la probabilidad, utilizaremos la función:

$$p(w_1 ... w_k) = \prod_{i=1}^k p(w_i|w_{i-1} ... w_{i-n+1})$$

Dado que las cadenas pueden extenderse y las probabilidades son pequeñas, es posible que la probabilidad se haga tan pequeña que aparezca como un cero. Para evitar esto, utilizaremos probabilidad logarítimicada, dada por:

$$\log p(w_1 ... w_k) = \sum_{i=1}^k \log p(w_i|w_{i-1} ... w_{i-n+1})$$

### 1. Obtener la probabilidad de una cadena

In [41]:
def get_sent_probability(sentence: str, vocab: defaultdict, model: tuple) -> float:
    A, Pi = model
    # Getting the n from n-grams
    n = len(A.shape)
    indexed_sentence = [vocab[word] for word in sentence.split()]
    first_indexed_word = indexed_sentence[0]
    # Getting initial probability
    try:
        probability = np.log(Pi[first_indexed_word])
    except:
        print(f"[WARN] OOV for word as BOS with index={first_indexed_word}")
        probability = 0.0

    # Getting n-grams of the sentence
    n_grams = get_ngrams([indexed_sentence], n)
    for n_gram in n_grams:
        try:
          probability += np.log(A[n_gram])
        except:
          print(f"[WARN] OOV for n_gram={n_gram}")
          probability += 0.0

    return probability

In [45]:
sentence = test_data[0]
print(f"La probabilidad de la cadena: <{sentence}>")
print(f"\t\t Modelo de bigramas: ", np.exp(get_sent_probability(sentence, kolo_vocab, bigram_model)))

La probabilidad de la cadena: <favor de brincar este corral >
[WARN] OOV for word as BOS with index=122
[WARN] OOV for n_gram=(122, 71)
		 Modelo de bigramas:  2.0348094858748592e-06


In [46]:
sentence = test_data[-1]
print(f"La probabilidad de la cadena: <{sentence}>")
print(f"\t\t Modelo de bigramas: ", np.exp(get_sent_probability(sentence, kolo_vocab, bigram_model)))

La probabilidad de la cadena: < y yo no llegaré a mi casa esta tarde>
		 Modelo de bigramas:  1.203610091611816e-13


In [80]:
TEST_SENTENCE = "el marrano no"

In [81]:
np.exp(get_sent_probability(TEST_SENTENCE, kolo_vocab, bigram_model))

0.0003780713231551132

In [82]:
A_bigram[0][2]

0.007575757575757576

### 2. Predecir la palabra siguiente

In [109]:
def predict_next_word(sentence: str, vocab: defaultdict, vocab_words: dict, model: tuple) -> str:
    A, Pi = model
    history = len(A.shape) - 1
    indexed_sentence = [vocab[word] for word in sentence.split()]
    prev_n_gram = tuple(indexed_sentence[-history:])
    probability = get_sent_probability(sentence, vocab, model)
    next_word = np.argmax(probability + np.log(A[prev_n_gram]))
    return vocab_words[next_word]

In [110]:
predict_next_word(TEST_SENTENCE, kolo_vocab, kolo_words_idx, bigram_model)

'quiere'

### 3. Generación de texto

Iterando sobre la función anterior podemos producir texto. Nustro algoritmo buscara el token *EOS* para detenerse o despues de producir *N* tokens.

In [111]:
def generate_laguage(sentence: str, vocab: defaultdict, vocab_words: dict, model: tuple, limit: int) -> str:
    next_word = ""
    result = sentence
    i = 0
    while next_word != EOS:
        next_word = predict_next_word(result, vocab, vocab_words, model)
        result += " " + next_word
        i += 1
        if i == limit:
            break

    return result

In [112]:
print(f"Modelo de bigramas: {TEST_SENTENCE}")
generate_laguage(TEST_SENTENCE, kolo_vocab, kolo_words_idx, bigram_model, 100)

Modelo de bigramas: el marrano no


'el marrano no quiere brincar el corral <EOS>'